# 07b. Avaliar predictions — atribuição mãe + endereço

Lê o txt pré-processado no bash (filtro `p ≥ 0,25`, 6 colunas, dedupe),
aplica a escada do [`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb)
e métricas no espírito do [`05_avaliar_mae_endereco.ipynb`](05_avaliar_mae_endereco.ipynb)
/ [`06_numeros_finais.ipynb`](06_numeros_finais.ipynb).

Formato do `PRED_TXT` (espaço, sem header):

`status x id1 id2 total_score match_probability`

- pré-etapa bash: corte do piso, drop das `s*`, dedupe por `id1,id2` (maior `p`)
- checkpoints parquet em `OUTPUT_DIR` (`REBUILD_*=True` recalcula; `False` retoma)
- **ids = `id_seq` do** [`07_exportar_csv_aplicacao_splink.ipynb`](07_exportar_csv_aplicacao_splink.ipynb)
  → depara com o mesmo `ROW_NUMBER() OVER (ORDER BY unique_id)` nas bases Luis V3

Ajuste `PRED_TXT` (e paths Luis, se mudarem) na célula de config.


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_ARQUIVO,
    LISTA_OURO_AVALIACAO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    OUTPUT_DIR,
    SPLINK_INPUT_VIEW,
    cpf_norm_sql,
    drop_splink_temp_tables,
    ensure_output_dir,
    export_parquet,
    get_connection,
    lista_ouro_colunas,
    load_table_from_parquet,
    materialize_splink_input,
    print_paths,
    require_input,
)

# --- config ---
PRED_TXT = Path(
    '/data/singed/ramon.goncalves/probabilistico_output/censo_cpf/'
    'cpareia_results/pred_gz_p025_dedup.txt'
)  # ajuste

CENSO_V3_LUIS = Path(
    '/data/singed/capefe/scripts_luis/Pareamento Probabilístico/'
    'V3 20260925/censo_limpo.parquet'
)
CPF_V3_LUIS = Path(
    '/data/singed/capefe/scripts_luis/Pareamento Probabilístico/'
    'V3 20260925/cpf_limpo.parquet'
)

PISO_MODELO = 0.90
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF

# True = recalcula e grava; False = lê parquet se existir
REBUILD_DEPARA = False
REBUILD_PRED = False
REBUILD_PARES = False
REBUILD_ATRIB = False

DEPARA_CENSO_OUT = OUTPUT_DIR / '07b_depara_censo.parquet'
DEPARA_CPF_OUT = OUTPUT_DIR / '07b_depara_cpf.parquet'
CENSO_LUIS_OUT = OUTPUT_DIR / '07b_censo_luis_input.parquet'
CPF_LUIS_OUT = OUTPUT_DIR / '07b_cpf_luis_input.parquet'
PRED_MAPPED_OUT = OUTPUT_DIR / '07b_pred_mapped.parquet'
PARES_OUT = OUTPUT_DIR / '07b_pares.parquet'
ATRIB_OUT = OUTPUT_DIR / 'splink_atribuicao_mae_endereco_gz.parquet'
COBERTURA_UF_OUT = OUTPUT_DIR / '07b_cobertura_uf.parquet'

print_paths()
ensure_output_dir()
print('PRED_TXT:', PRED_TXT)
print('CENSO_V3_LUIS:', CENSO_V3_LUIS)
print('CPF_V3_LUIS:', CPF_V3_LUIS)
print('REBUILD:', {
    'depara': REBUILD_DEPARA,
    'pred': REBUILD_PRED,
    'pares': REBUILD_PARES,
    'atrib': REBUILD_ATRIB,
})
print('Atribuição (saída):', ATRIB_OUT)

if REBUILD_PRED or not PRED_MAPPED_OUT.exists():
    require_input(PRED_TXT, label='PRED_TXT')
require_input(CENSO_V3_LUIS, label='CENSO_V3_LUIS')
require_input(CPF_V3_LUIS, label='CPF_V3_LUIS')
require_input(LISTA_OURO_AVALIACAO_ARQUIVO, label='LISTA_OURO_AVALIACAO')
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')

con = get_connection()
drop_splink_temp_tables(con)
print(
    f'Faixas: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | '
    f'baixo>={PISO_BAIXO} | teto={TETO}'
)


## Depara `id_seq` → chave real (Luis V3)

Reproduz o recorte e o `ROW_NUMBER() OVER (ORDER BY unique_id)` do
[`07_exportar_csv_aplicacao_splink.ipynb`](07_exportar_csv_aplicacao_splink.ipynb).
Monta tabelas Splink (`censo_` + id / `cpf_` + cpf) para a escada.
Checkpoint: `07b_depara_*.parquet` / `07b_*_luis_input.parquet`.


In [ ]:
censo_path = str(CENSO_V3_LUIS).replace(chr(39), chr(39) * 2)
cpf_path = str(CPF_V3_LUIS).replace(chr(39), chr(39) * 2)
ouro_path = str(LISTA_OURO_AVALIACAO_ARQUIVO).replace(chr(39), chr(39) * 2)

ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{ouro_path}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro_avaliacao AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{ouro_path}')
WHERE {id_ouro} IS NOT NULL OR {cpf_ouro} IS NOT NULL
''')

_pode_load_depara = (
    not REBUILD_DEPARA
    and DEPARA_CENSO_OUT.exists()
    and DEPARA_CPF_OUT.exists()
    and CENSO_LUIS_OUT.exists()
    and CPF_LUIS_OUT.exists()
)

if _pode_load_depara:
    load_table_from_parquet(con, 'depara_censo', DEPARA_CENSO_OUT)
    load_table_from_parquet(con, 'depara_cpf', DEPARA_CPF_OUT)
    load_table_from_parquet(con, 'censo_luis_input', CENSO_LUIS_OUT)
    load_table_from_parquet(con, 'cpf_luis_input', CPF_LUIS_OUT)
    print('depara/input: lido do parquet')
else:
    con.execute(f'''
    CREATE OR REPLACE TABLE censo_export AS
    SELECT *
    FROM read_parquet('{censo_path}') a
    WHERE CAST(a.id_morador AS VARCHAR) IS NULL
       OR CAST(a.id_morador AS VARCHAR) NOT IN (
           SELECT id_censo FROM lista_ouro_avaliacao WHERE id_censo IS NOT NULL
       )
    ''')

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_export AS
    SELECT *
    FROM read_parquet('{cpf_path}') a
    WHERE {cpf_norm_sql('a.cpf')} IS NULL
       OR {cpf_norm_sql('a.cpf')} NOT IN (
           SELECT cpf_norm FROM lista_ouro_avaliacao WHERE cpf_norm IS NOT NULL
       )
    ''')

    print('censo_export:', f"{con.execute('SELECT COUNT(*) FROM censo_export').fetchone()[0]:,}")
    print('cpf_export:  ', f"{con.execute('SELECT COUNT(*) FROM cpf_export').fetchone()[0]:,}")

    dob_iso = (
        "strftime(coalesce("
        "try_strptime(trim(CAST(data_nascimento AS VARCHAR)), '%d/%m/%Y'), "
        "try_strptime(trim(CAST(data_nascimento AS VARCHAR)), '%Y-%m-%d')"
        "), '%Y-%m-%d')"
    )

    con.execute(f'''
    CREATE OR REPLACE TABLE depara_censo AS
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY unique_id) AS VARCHAR) AS id_seq,
        CAST(unique_id AS VARCHAR) AS unique_id_luis,
        CAST(id_morador AS VARCHAR) AS id_morador,
        'censo_' || CAST(id_morador AS VARCHAR) AS unique_id_censo,
        CAST(nome_fonetico AS VARCHAR) AS nome_completo_phon,
        CAST(fon_primeiro AS VARCHAR) AS primeiro_nome_phon,
        CAST(fon_ultimo AS VARCHAR) AS ultimo_nome_phon,
        {dob_iso} AS data_nascimento,
        CAST(idade AS INTEGER) AS idade,
        CAST(nome_fonetico_mae AS VARCHAR) AS nome_mae_phon,
        CAST(logradouro AS VARCHAR) AS logradouro_norm,
        CAST(cep AS VARCHAR) AS cep,
        CAST(cod_mun AS VARCHAR) AS cod_municipio,
        CAST(sexo AS VARCHAR) AS sexo
    FROM censo_export
    ''')

    con.execute(f'''
    CREATE OR REPLACE TABLE depara_cpf AS
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY unique_id) AS VARCHAR) AS id_seq,
        CAST(unique_id AS VARCHAR) AS unique_id_luis,
        {cpf_norm_sql('cpf')} AS cpf_norm,
        'cpf_' || {cpf_norm_sql('cpf')} AS unique_id_cpf,
        CAST(nome_fonetico AS VARCHAR) AS nome_completo_phon,
        CAST(fon_primeiro AS VARCHAR) AS primeiro_nome_phon,
        CAST(fon_ultimo AS VARCHAR) AS ultimo_nome_phon,
        {dob_iso} AS data_nascimento,
        CAST(idade AS INTEGER) AS idade,
        CAST(nome_fonetico_mae AS VARCHAR) AS nome_mae_phon,
        CAST(logradouro AS VARCHAR) AS logradouro_norm,
        CAST(cep AS VARCHAR) AS cep,
        CAST(cod_mun AS VARCHAR) AS cod_municipio,
        CAST(sexo AS VARCHAR) AS sexo
    FROM cpf_export
    ''')

    con.execute('''
    CREATE OR REPLACE TABLE censo_luis_input AS
    SELECT
        unique_id_censo AS unique_id,
        id_morador AS person_id_censo,
        nome_completo_phon,
        primeiro_nome_phon,
        ultimo_nome_phon,
        data_nascimento,
        idade,
        nome_mae_phon,
        logradouro_norm,
        cep,
        cod_municipio,
        sexo,
        CAST(NULL AS VARCHAR) AS cpf_norm,
        'censo' AS origem
    FROM depara_censo
    ''')

    con.execute('''
    CREATE OR REPLACE TABLE cpf_luis_input AS
    SELECT
        unique_id_cpf AS unique_id,
        CAST(NULL AS VARCHAR) AS person_id_censo,
        nome_completo_phon,
        primeiro_nome_phon,
        ultimo_nome_phon,
        data_nascimento,
        idade,
        nome_mae_phon,
        logradouro_norm,
        cep,
        cod_municipio,
        sexo,
        cpf_norm,
        'cpf' AS origem
    FROM depara_cpf
    ''')

    export_parquet(con, 'depara_censo', path=DEPARA_CENSO_OUT)
    export_parquet(con, 'depara_cpf', path=DEPARA_CPF_OUT)
    export_parquet(con, 'censo_luis_input', path=CENSO_LUIS_OUT)
    export_parquet(con, 'cpf_luis_input', path=CPF_LUIS_OUT)
    print('depara/input: exportado')

materialize_splink_input(
    con,
    censo_table='censo_luis_input',
    cpf_table='cpf_luis_input',
)

display(con.execute('''
SELECT id_seq, id_morador, unique_id_censo, data_nascimento, nome_completo_phon
FROM depara_censo LIMIT 3
''').df())
display(con.execute('''
SELECT id_seq, cpf_norm, unique_id_cpf, data_nascimento, nome_completo_phon
FROM depara_cpf LIMIT 3
''').df())
print('depara_censo:', f"{con.execute('SELECT COUNT(*) FROM depara_censo').fetchone()[0]:,}")
print('depara_cpf:  ', f"{con.execute('SELECT COUNT(*) FROM depara_cpf').fetchone()[0]:,}")


## Carga do `PRED_TXT`

Só roda se `REBUILD_PRED` ou se ainda não existe `07b_pred_mapped.parquet`.
DuckDB lê o txt dedupado (6 colunas).


In [ ]:
if not REBUILD_PRED and PRED_MAPPED_OUT.exists():
    print('pred_raw: pulado (usando pred_mapped parquet)')
else:
    pred_path = str(PRED_TXT).replace(chr(39), chr(39) * 2)

    con.execute(f'''
    CREATE OR REPLACE TABLE pred_raw AS
    SELECT
        CAST(status AS VARCHAR) AS status,
        CAST(x AS VARCHAR) AS x,
        CAST(id1 AS VARCHAR) AS id1,
        CAST(id2 AS VARCHAR) AS id2,
        TRY_CAST(total_score AS DOUBLE) AS total_score,
        TRY_CAST(match_probability AS DOUBLE) AS match_probability,
        '{PRED_TXT.name}' AS arquivo
    FROM read_csv(
        '{pred_path}',
        delim=' ',
        header=false,
        auto_detect=false,
        columns={{
            'status': 'VARCHAR',
            'x': 'VARCHAR',
            'id1': 'VARCHAR',
            'id2': 'VARCHAR',
            'total_score': 'VARCHAR',
            'match_probability': 'VARCHAR'
        }}
    )
    WHERE TRY_CAST(match_probability AS DOUBLE) >= {PISO_BAIXO}
    ''')

    print(f'PRED_TXT: {PRED_TXT} | piso p>={PISO_BAIXO}')
    display(con.execute('SELECT * FROM pred_raw LIMIT 5').df())
    display(con.execute('''
    SELECT
        COUNT(*) AS n,
        COUNT(DISTINCT id1 || '|' || id2) AS n_pares_distintos,
        COUNT(DISTINCT arquivo) AS n_arquivos
    FROM pred_raw
    ''').df())


## Mapa `id_seq` → `unique_id`

Txt já vem dedupado: só strip de prefixo (se houver) e join no depara.
Checkpoint: `07b_pred_mapped.parquet`.


In [ ]:
_tipo = con.execute(
    "SELECT table_type FROM information_schema.tables "
    "WHERE table_schema = 'main' AND table_name = 'splink_predictions'"
).fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')

if not REBUILD_PRED and PRED_MAPPED_OUT.exists():
    load_table_from_parquet(con, 'pred_mapped', PRED_MAPPED_OUT)
    print('pred_mapped: lido do parquet')
else:
    id1_seq = (
        "CASE WHEN id1 LIKE 'censo_%' THEN substr(id1, 7) ELSE id1 END"
    )
    id2_seq = (
        "CASE WHEN id2 LIKE 'cpf_%' THEN substr(id2, 5) ELSE id2 END"
    )

    con.execute(f'''
    CREATE OR REPLACE TABLE pred_mapped AS
    SELECT
        dc.unique_id_censo,
        dp.unique_id_cpf,
        r.match_probability,
        r.total_score,
        r.status,
        r.x,
        r.arquivo,
        r.id1,
        r.id2
    FROM (
        SELECT
            *,
            {id1_seq} AS id_seq_censo,
            {id2_seq} AS id_seq_cpf
        FROM pred_raw
        WHERE match_probability IS NOT NULL
    ) r
    LEFT JOIN depara_censo dc ON dc.id_seq = r.id_seq_censo
    LEFT JOIN depara_cpf dp ON dp.id_seq = r.id_seq_cpf
    ''')

    export_parquet(con, 'pred_mapped', path=PRED_MAPPED_OUT)
    print('pred_mapped: exportado →', PRED_MAPPED_OUT)

con.execute('''
CREATE OR REPLACE TABLE splink_predictions AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM pred_mapped
WHERE unique_id_censo IS NOT NULL AND unique_id_cpf IS NOT NULL
''')

n_mapped = con.execute('SELECT COUNT(*) FROM pred_mapped').fetchone()[0]
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
n_sem_mapa = con.execute('''
SELECT COUNT(*) FROM pred_mapped
WHERE unique_id_censo IS NULL OR unique_id_cpf IS NULL
''').fetchone()[0]

print(f'mapped: {n_mapped:,} → com depara: {n_pred:,}')
print(f'Sem mapa (id_seq fora do depara): {n_sem_mapa:,}')

display(con.execute(f'''
SELECT
    COUNT(*) AS n_pred,
    COUNT(*) FILTER (
        WHERE ca.unique_id IS NOT NULL AND pb.unique_id IS NOT NULL
    ) AS n_com_join_input,
    COUNT(*) FILTER (WHERE ca.unique_id IS NULL) AS n_censo_fora_input,
    COUNT(*) FILTER (WHERE pb.unique_id IS NULL) AS n_cpf_fora_input
FROM splink_predictions p
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
''').df())

display(con.execute('''
SELECT
    CASE
        WHEN match_probability >= 0.90 THEN 'p_ge_090'
        WHEN match_probability >= 0.50 THEN 'p_050_090'
        WHEN match_probability >= 0.25 THEN 'p_025_050'
        ELSE 'p_lt_025'
    END AS faixa_p,
    COUNT(*) AS n
FROM splink_predictions
GROUP BY 1
ORDER BY 1 DESC
''').df())


## Pares elegíveis ao resgate (R1 / R2 / R3)

Mesma lógica do 04. Checkpoint: `07b_pares.parquet`.


In [ ]:
if not REBUILD_PARES and PARES_OUT.exists():
    load_table_from_parquet(con, 'pares', PARES_OUT)
    print('pares: lido do parquet |',
          f"{con.execute('SELECT COUNT(*) FROM pares').fetchone()[0]:,}")
    display(con.execute('''
    SELECT
        COUNT(*) FILTER (WHERE r1_medio) AS n_r1_medio,
        COUNT(*) FILTER (WHERE r2) AS n_r2,
        COUNT(*) FILTER (WHERE r3_medio) AS n_r3_medio,
        COUNT(*) FILTER (WHERE r1_baixo) AS n_r1_baixo,
        COUNT(*) FILTER (WHERE regra_medio IS NOT NULL) AS n_com_regra_medio,
        COUNT(*) FILTER (WHERE regra_baixo IS NOT NULL) AS n_com_regra_baixo
    FROM pares
    ''').df())
else:
    tok_ca = "string_split(ca.nome_completo_phon, ' ')"
    tok_pb = "string_split(pb.nome_completo_phon, ' ')"
    n_ca = f"len({tok_ca})"
    n_pb = f"len({tok_pb})"
    n_min = f"least({n_ca}, {n_pb})"
    n_inter = f"len(list_intersect({tok_ca}, {tok_pb}))"
    cobertura = f"({n_inter} * 1.0 / nullif({n_min}, 0))"
    contido = (
        f"{n_min} >= 2 AND {n_ca} <> {n_pb} AND {n_inter} = {n_min}"
    )
    primeiro_ok = (
        "ca.primeiro_nome_phon IS NOT NULL AND pb.primeiro_nome_phon IS NOT NULL "
        "AND ("
        "ca.primeiro_nome_phon = pb.primeiro_nome_phon "
        "OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) <= 1"
        ")"
    )
    data_igual = (
        "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
        "AND ca.data_nascimento = pb.data_nascimento"
    )
    data_ambas = (
        "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
        "AND ca.data_nascimento <> pb.data_nascimento"
    )
    delta_dias = (
        "abs(date_diff('day', "
        "CAST(ca.data_nascimento AS DATE), CAST(pb.data_nascimento AS DATE)))"
    )
    mesmo_dia_mes = (
        "day(CAST(ca.data_nascimento AS DATE)) = day(CAST(pb.data_nascimento AS DATE)) "
        "AND month(CAST(ca.data_nascimento AS DATE)) = month(CAST(pb.data_nascimento AS DATE))"
    )
    delta_ano = (
        "abs(year(CAST(ca.data_nascimento AS DATE)) "
        "- year(CAST(pb.data_nascimento AS DATE)))"
    )
    data_plausivel = (
        f"({delta_dias} <= 1) OR "
        f"(({mesmo_dia_mes}) AND ({delta_ano}) = 1) OR "
        f"(({mesmo_dia_mes}) AND ({delta_ano}) = 10)"
    )
    nome_muito_forte = (
        f"(ca.nome_completo_phon = pb.nome_completo_phon) OR ({contido})"
    )
    mae_presente = (
        "ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL"
    )
    jw_mae = "jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon)"

    r1_medio = (
        f"({data_igual}) AND ({primeiro_ok}) AND "
        f"(({contido}) OR ({cobertura} >= 0.75))"
    )
    r1_baixo = (
        f"({data_igual}) AND ({primeiro_ok}) AND "
        f"(({contido}) OR ({cobertura} >= 0.80))"
    )
    r2 = (
        f"({nome_muito_forte}) AND ({data_ambas}) AND ({data_plausivel})"
    )
    r3_medio = (
        f"({primeiro_ok}) AND ({cobertura} >= 0.55) AND ({mae_presente}) "
        f"AND ({jw_mae} >= 0.75)"
    )

    desempate = (
        "("
        "CAST(ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL AS INTEGER) "
        "+ CAST("
        "ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL "
        "AND ca.logradouro_norm = pb.logradouro_norm AS INTEGER) "
        "+ CAST("
        "ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)"
        ")"
    )

    con.execute(f'''
    CREATE OR REPLACE TABLE pares AS
    SELECT
        p.unique_id_censo,
        p.unique_id_cpf,
        p.match_probability,
        {desempate} AS desempate,
        ({r1_medio}) AS r1_medio,
        ({r1_baixo}) AS r1_baixo,
        ({r2}) AS r2,
        ({r3_medio}) AS r3_medio,
        CASE
            WHEN ({r1_medio}) THEN 'R1'
            WHEN ({r2}) THEN 'R2'
            WHEN ({r3_medio}) THEN 'R3'
        END AS regra_medio,
        CASE
            WHEN ({r1_baixo}) THEN 'R1'
            WHEN ({r2}) THEN 'R2'
        END AS regra_baixo
    FROM splink_predictions p
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
    WHERE p.match_probability >= {PISO_BAIXO}
    ''')

    n_pares = con.execute('SELECT COUNT(*) FROM pares').fetchone()[0]
    print('Pares com p >=', PISO_BAIXO, 'e join no input:', f'{n_pares:,}')
    display(con.execute('''
    SELECT
        COUNT(*) FILTER (WHERE r1_medio) AS n_r1_medio,
        COUNT(*) FILTER (WHERE r2) AS n_r2,
        COUNT(*) FILTER (WHERE r3_medio) AS n_r3_medio,
        COUNT(*) FILTER (WHERE r1_baixo) AS n_r1_baixo,
        COUNT(*) FILTER (WHERE regra_medio IS NOT NULL) AS n_com_regra_medio,
        COUNT(*) FILTER (WHERE regra_baixo IS NOT NULL) AS n_com_regra_baixo
    FROM pares
    ''').df())
    export_parquet(con, 'pares', path=PARES_OUT)
    print('pares: exportado →', PARES_OUT)


## Atribuição por faixa

Três passos iguais ao 04: modelo → resgate → baixo.
Checkpoint: `splink_atribuicao_mae_endereco_gz.parquet`.


In [ ]:
if not REBUILD_ATRIB and ATRIB_OUT.exists():
    load_table_from_parquet(con, 'atribuicao', ATRIB_OUT)
    print('atribuicao: lida do parquet |',
          f"{con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]:,}")
else:
    con.execute('''
    CREATE OR REPLACE TABLE atribuicao (
        unique_id_censo VARCHAR,
        unique_id_cpf VARCHAR,
        match_probability DOUBLE,
        faixa VARCHAR,
        regra VARCHAR
    )
    ''')

    faixas = [
        ('modelo', PISO_MODELO, None, None),
        ('resgate', PISO_RESGATE, PISO_MODELO, 'regra_medio'),
        ('baixo', PISO_BAIXO, PISO_RESGATE, 'regra_baixo'),
    ]

    for nome_faixa, piso, teto_nota, col_regra in faixas:
        if teto_nota is None:
            filtro_p = f'match_probability >= {piso}'
        else:
            filtro_p = (
                f'match_probability >= {piso} '
                f'AND match_probability < {teto_nota}'
            )
        if col_regra is None:
            filtro_regra = 'TRUE'
            regra_sql = "'score'"
        else:
            filtro_regra = f'{col_regra} IS NOT NULL'
            regra_sql = col_regra

        con.execute(f'''
        CREATE OR REPLACE TABLE faixa_base AS
        SELECT
            unique_id_censo,
            unique_id_cpf,
            match_probability,
            desempate,
            {regra_sql} AS regra
        FROM pares
        WHERE {filtro_p}
          AND {filtro_regra}
          AND unique_id_censo NOT IN (SELECT unique_id_censo FROM atribuicao)
          AND unique_id_cpf NOT IN (
              SELECT unique_id_cpf
              FROM atribuicao
              GROUP BY 1
              HAVING COUNT(*) >= {TETO}
          )
        ''')

        con.execute('''
        CREATE OR REPLACE TABLE faixa_melhor AS
        SELECT unique_id_censo, unique_id_cpf, match_probability, regra
        FROM faixa_base
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY unique_id_censo
            ORDER BY match_probability DESC, desempate DESC, unique_id_cpf
        ) = 1
        ''')

        antes = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
        con.execute(f'''
        INSERT INTO atribuicao
        SELECT
            m.unique_id_censo,
            m.unique_id_cpf,
            m.match_probability,
            '{nome_faixa}' AS faixa,
            m.regra
        FROM faixa_melhor m
        JOIN (
            SELECT unique_id_cpf, COUNT(*) AS n_novo
            FROM faixa_melhor
            GROUP BY 1
        ) g ON g.unique_id_cpf = m.unique_id_cpf
        LEFT JOIN (
            SELECT unique_id_cpf, COUNT(*) AS n_ja
            FROM atribuicao
            GROUP BY 1
        ) j ON j.unique_id_cpf = m.unique_id_cpf
        WHERE coalesce(j.n_ja, 0) + g.n_novo <= {TETO}
        ''')
        depois = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
        print(f'{nome_faixa} ({filtro_p}): +{depois - antes:,} (total {depois:,})')
    export_parquet(con, 'atribuicao', path=ATRIB_OUT)
    print('atribuicao: exportada →', ATRIB_OUT)


## Cobertura


In [ ]:
n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_ok = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
print(f'Censo aplicação: {n_app:,}')
print(f'Atribuídos: {n_ok:,} ({100.0 * n_ok / n_app:.1f}%)')

display(con.execute('''
SELECT
    faixa,
    regra,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao
GROUP BY 1, 2
ORDER BY
    CASE faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END,
    regra
''').df())

display(con.execute('''
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
''').df())



## Cobertura por UF

Universo = Censo Luis V3 completo. Determinístico = lista ouro (removidos no 07).
Probabilístico = `atribuicao`. `total_pareado` = soma (disjuntos). `%` sobre o universo.


In [ ]:
censo_path = str(CENSO_V3_LUIS).replace(chr(39), chr(39) * 2)

con.execute(f'''
CREATE OR REPLACE TABLE cobertura_uf AS
WITH universo AS (
    SELECT
        CAST(id_morador AS VARCHAR) AS id_morador,
        substr(CAST(cod_mun AS VARCHAR), 1, 2) AS uf
    FROM read_parquet('{censo_path}')
    WHERE CAST(id_morador AS VARCHAR) IS NOT NULL
),
det AS (
    SELECT DISTINCT id_censo AS id_morador
    FROM lista_ouro_avaliacao
    WHERE id_censo IS NOT NULL
),
prob AS (
    SELECT DISTINCT
        CASE
            WHEN a.unique_id_censo LIKE 'censo_%'
                THEN substr(a.unique_id_censo, 7)
            ELSE a.unique_id_censo
        END AS id_morador
    FROM atribuicao a
)
SELECT
    u.uf,
    COUNT(*) FILTER (WHERE d.id_morador IS NOT NULL) AS deterministico,
    COUNT(*) FILTER (WHERE p.id_morador IS NOT NULL) AS probabilistico,
    COUNT(*) FILTER (
        WHERE d.id_morador IS NOT NULL OR p.id_morador IS NOT NULL
    ) AS total_pareado,
    COUNT(*) AS total_existente,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE d.id_morador IS NOT NULL OR p.id_morador IS NOT NULL
        ) / nullif(COUNT(*), 0),
        2
    ) AS pct
FROM universo u
LEFT JOIN det d ON d.id_morador = u.id_morador
LEFT JOIN prob p ON p.id_morador = u.id_morador
GROUP BY 1
ORDER BY 1
''')

display(con.execute('SELECT * FROM cobertura_uf').df())
display(con.execute('''
SELECT
    SUM(deterministico) AS deterministico,
    SUM(probabilistico) AS probabilistico,
    SUM(total_pareado) AS total_pareado,
    SUM(total_existente) AS total_existente,
    ROUND(
        100.0 * SUM(total_pareado) / nullif(SUM(total_existente), 0),
        2
    ) AS pct
FROM cobertura_uf
''').df())

export_parquet(con, 'cobertura_uf', path=COBERTURA_UF_OUT)
print('Exportado:', COBERTURA_UF_OUT)


## Qualidade dos pares atribuídos

Acordo geral e por faixa; distribuição de `p`; consistência de UF (censo vs CPF).
Sem ground truth — diagnósticos de plausibilidade.


In [ ]:
# Geral
display(con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
    ) / COUNT(*), 1) AS pct_mae_preenchida,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE substr(ca.cod_municipio, 1, 2) = substr(pb.cod_municipio, 1, 2)
    ) / COUNT(*), 1) AS pct_uf_igual
FROM atribuicao a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
''').df())

# Por faixa: acordo + UF
display(con.execute(f'''
SELECT
    a.faixa,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE substr(ca.cod_municipio, 1, 2) = substr(pb.cod_municipio, 1, 2)
    ) / COUNT(*), 1) AS pct_uf_igual
FROM atribuicao a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
GROUP BY 1
ORDER BY
    CASE a.faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END
''').df())

# Nota (p) dos atribuídos por faixa
display(con.execute('''
SELECT
    faixa,
    COUNT(*) AS n,
    ROUND(MIN(match_probability), 4) AS p_min,
    ROUND(quantile_cont(match_probability, 0.10), 4) AS p_p10,
    ROUND(quantile_cont(match_probability, 0.50), 4) AS p_p50,
    ROUND(quantile_cont(match_probability, 0.90), 4) AS p_p90,
    ROUND(MAX(match_probability), 4) AS p_max
FROM atribuicao
GROUP BY 1
ORDER BY
    CASE faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END
''').df())


## Quem ficou de fora

Melhor par no predict por Censo sem atribuição.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE melhor_qualquer AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        CAST(
            ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
            AS INTEGER
        ) DESC,
        CAST(
            ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
            AND ca.logradouro_norm = pb.logradouro_norm
            AS INTEGER
        ) DESC,
        CAST(
            ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            AS INTEGER
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id AS unique_id_censo
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute('''
CREATE OR REPLACE TABLE sem_cpf_melhor AS
SELECT
    s.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    CASE
        WHEN m.match_probability IS NULL THEN 'sem_par'
        WHEN m.match_probability >= 0.90 THEN 'p_ge_090'
        WHEN m.match_probability >= 0.50 THEN 'p_050_090'
        WHEN m.match_probability >= 0.25 THEN 'p_025_050'
        ELSE 'p_lt_025'
    END AS faixa_p
FROM censo_sem_cpf s
LEFT JOIN melhor_qualquer m ON m.unique_id_censo = s.unique_id_censo
''')

n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]
assert n_ok + n_sem == n_app

display(con.execute(f'''
SELECT
    a.faixa_p,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif({n_sem}, 0), 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'p_ge_090'),
    (2, 'p_050_090'),
    (3, 'p_025_050'),
    (4, 'p_lt_025'),
    (5, 'sem_par')
) a(ord, faixa_p)
LEFT JOIN (
    SELECT faixa_p, COUNT(*) AS n
    FROM sem_cpf_melhor
    GROUP BY 1
) g ON g.faixa_p = a.faixa_p
ORDER BY a.ord
''').df())


In [ ]:
con.close()
print('ok')
